# 1. CREATING TABLE

## Adding Constraints

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cpt_utility_catalog.gold.fct_dam_levels(
    dam_level_key BIGINT NOT NULL,
    date_key INT NOT NULL,
    height_m DECIMAL(5,2),
    storage_Ml DECIMAL(10,3),
    current_pct DECIMAL(12,8),
    last_year_current_pct DECIMAL(12,8),

    CONSTRAINT pk_fct_dam_levels PRIMARY KEY (dam_level_key) RELY,
    CONSTRAINT fk_fct_dam_levels_date FOREIGN KEY (date_key) REFERENCES cpt_utility_catalog.gold.dim_date(date_key) RELY,
    CONSTRAINT fk_fct_dam_levels_dam FOREIGN KEY (dam_key) REFERENCES cpt_utility_catalog.gold.dim_dam(dam_key) RELY
)

## 1.2 Populating Table

In [0]:
%sql
INSERT OVERWRITE TABLE cpt_utility_catalog.gold.fct_dam_levels
SELECT
xxhash64(d.id) AS dam_level_key,

COALESCE(CAST(date_format(d.date,'yyyyMMdd') AS INT), -1) AS date_key,
COALESCE(dd.dam_key, xxhash64('unmapped')) AS dam_key,

d.height_m,
d.storage_Ml,
d.current_pct,
d.last_year_pct AS last_year_current_pct


FROM cpt_utility_catalog.silver.silver_dam_levels_cleaned d

LEFT JOIN cpt_utility_catalog.gold.dim_dam dd ON xxhash64(LOWER(TRIM(d.dam_name))) = dd.dam_key

WHERE d.dam_name != 'total_stored_big_6'
